In [1]:
import torch
import torch.nn as nn
import pickle
import pandas as pd
import numpy as np

In [2]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__() 
        self.model = nn.Sequential(
            nn.Linear(12, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),

            
            nn.Linear(64, 1)           
                                    
            
        )

    def forward(self, x):
        return self.model(x)
        
    
    

In [3]:
model = ANN()

In [4]:
## Load Model
model.load_state_dict(torch.load("ann_model.pth"))

## Load pickle files
with open("label_encoder_gender.pkl", "rb") as file:
    label_encoder_gender = pickle.load(file)
with open("onehot_encoder_geo.pkl", "rb") as file:
    onehot_encoder_geo = pickle.load(file)
with open("scaler.pkl", "rb") as file:
    scaler = pickle.load(file)




In [5]:
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}


In [6]:
geo_encoded = onehot_encoder_geo.transform([[input_data["Geography"]]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns = onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

D:\Anaconda\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [7]:
input_df = pd.DataFrame([input_data])


In [8]:
input_data = pd.concat([input_df.reset_index(drop = True), geo_encoded_df], axis = 1)
input_data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,France,Male,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [9]:
input_df["Gender"] = label_encoder_gender.transform(input_df["Gender"])

In [10]:
input_df = pd.concat([input_df.drop("Geography", axis = 1), geo_encoded_df], axis = 1 )

In [11]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [12]:
input_df_scaled = scaler.transform(input_df)

In [13]:
input_tensor = torch.tensor(input_df_scaled, dtype = torch.float32)

In [14]:
## Predict Churn

model.eval()
with torch.no_grad():
    prediction = model(input_tensor)

pred_probs = torch.sigmoid(prediction).item()

In [15]:
pred_probs

0.15186502039432526

In [16]:
if pred_probs>=0.5:
    print("Customer is Likely to churn")

else:
    print("Customer is not likely to churn")

Customer is not likely to churn
